# Metadata overview

Filters the scBaseCount sample metadata to a lung subset and produces the three artifacts that feed downstream notebooks:

- `output/metadata/datasets.csv` (full lung intersection, consumed by `clustering.ipynb`)
- `output/metadata/accession_disease_categories.json` (per-accession `DISEASE_MAP` labels for the lung intersection)
- `output/metadata/datasets_subset_qc.csv` (per-cohort QC-passing sample of up to a certain number of accessions for cytetype evaluation)

The lung intersection is built by intersecting a lung-tissue regex with a lung-disease regex on `sample_metadata.parquet`, after dropping samples with fewer than `minObsCount` cells and any sample matched by `NORMAL_HEALTHY_RE` (healthy controls, unknowns, and explicit negations like `no COPD` or `Non-disease`). See `scripts/metadata/README.md` for the full regex definitions.

In [ ]:
from pathlib import Path

import pandas as pd
from metadata import (
    MetadataConfig,
    QcThresholds,
    apply_qc,
    compute_obs_qc,
    export_accession_disease_categories,
    export_datasets,
    filter_lung,
    load_sample,
    most_specific_disease_label,
    obs_rows_for_srx,
)
from metadata.viz import plot_cell_count_distribution, plot_disease_breakdown, plot_sample_breakdown
from shared.repo import REPO_ROOT

ROOT = REPO_ROOT

## Config

In [ ]:
_DATA_DIR = ROOT / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens"

cfg = MetadataConfig(
    sampleParquetPath=_DATA_DIR / "scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_sample_metadata.parquet",
    obsParquetPath=_DATA_DIR / "scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_obs_metadata.parquet",
    minObsCount=1000,
    outputDir=ROOT / "output/metadata",
)

## Load and Filter

In [ ]:
sample = load_sample(cfg)
result = filter_lung(sample, cfg)
print(f"Loaded {len(sample):,} samples")

## Summary

In [ ]:
# summary counts
n_total = len(sample)
n_floor = (sample["obs_count"] < cfg.minObsCount).sum()
n_known = len(result.sampleKnown)
n_inter = len(result.lungIntersection)
n_cancer = len(result.lungIntersectionCancer)

pd.DataFrame(
    [
        {"subset": "total samples", "n": n_total},
        {"subset": f"dropped (< {cfg.minObsCount} cells)", "n": n_floor},
        {"subset": "known disease + tissue", "n": n_known},
        {"subset": "lung intersection (disease AND tissue)", "n": n_inter},
        {"subset": "lung intersection - cancer only", "n": n_cancer},
    ]
)

## Visualisations

In [ ]:
plot_sample_breakdown(sample, result, figs_dir=ROOT / "output/metadata/figs")

In [ ]:
plot_disease_breakdown(result, figs_dir=ROOT / "output/metadata/figs")

In [ ]:
plot_cell_count_distribution(result, figs_dir=ROOT / "output/metadata/figs")

## Export

In [ ]:
datasets_path = export_datasets(result, cfg)
print(f"datasets.csv: {datasets_path}")

## Disease Categories

For every accession in `result.lungIntersection`, list each `DISEASE_MAP` label whose regex matches the raw `disease` string. Output: `output/metadata/accession_disease_categories.json`.

`DISEASE_MAP` mixes two kinds of labels:

- A nested lung-cancer subtree (`Lung Cancer` -> `SCLC` / `NSCLC` -> `LUAD` / `LUSC` / `LCC`) where parent and child both match the same accession on purpose. For example a row whose `disease` is `lung adenocarcinoma` matches `Lung Cancer`, `NSCLC`, and `LUAD` simultaneously.
- A flat set of cohort labels (`IPF / Pulmonary Fibrosis`, `COVID-19 / SARS-CoV-2`, `COPD`, `Cystic Fibrosis`, `Interstitial Lung Disease`, `Pulmonary Hypertension`) which are siblings of each other. Cross-cohort overlap is essentially absent: only 5 of the ~800 lung-intersection accessions match more than one of these (a few `lung cancer, COPD` and one `COVID-19, IPF` row).

Two special-case behaviours are worth knowing about:

- Rows whose `disease` string contains `non-CF` or `non-cystic fibrosis` (pure non-CF controls and mixed `cystic fibrosis (CF) and non-CF` datasets) are stripped of the `Cystic Fibrosis` label inside `disease_categories_for`, since they do not represent a CF cohort. Those rows fall into `Other`.
- `NORMAL_HEALTHY_RE` is applied earlier in `filter_lung`, so explicit negations such as `no COPD`, `Non-disease`, or `no diagnosed disease` are already excluded from `result.lungIntersection` and never reach the category step.

In [ ]:
categories_path = export_accession_disease_categories(result.lungIntersection, cfg)
print(f"accession_disease_categories.json: {categories_path}")

## QC + Cytetype Dataset

Build the cytetype pipeline input (`output/metadata/datasets_subset_qc.csv`). For each of the five non-cancer lung-disease cohorts (`IPF`, `COVID-19`, `COPD`, `Interstitial Lung Disease`, `Cystic Fibrosis`), compute per-accession median gene/UMI QC metrics from the obs parquet, drop accessions below `QcThresholds`, and sample at most `SAMPLE_SIZE` accessions per cohort from the QC-passing pool with a fixed `RANDOM_STATE`. The CSV row schema is `srx_accession, file_path, obs_count, disease, diseaseLabel, medianGenesPerCell, medianUmisPerCell, nCellsForQc`. `disease` is the raw metadata string and `diseaseLabel` is the most-specific `DISEASE_MAP` match returned by `most_specific_disease_label` (`scripts/metadata/categorize.py`).

### Note on `cats[-1]` tie-break

`most_specific_disease_label` picks `cats[-1]` from `disease_categories_for`. Inside the lung-cancer subtree this is the most-specific child (LUAD/LUSC/LCC/SCLC/NSCLC, with Lung Cancer as the parent), which is the intended semantics. For cross-cohort comorbidities the choice is just whichever sibling appears later in `DISEASE_MAP`, which is an arbitrary tie-break. Across the 772-accession lung intersection only 5 rows are affected:

- `SRX25038161`, `SRX25038162`, `SRX25038168` (`lung cancer, COPD`) and `SRX25038163` (`lung cancer (patients with COPD)`) bucket as `COPD`
- `SRX10048373` (`COVID-19, idiopathic pulmonary fibrosis (IPF)`) buckets as `COVID-19 / SARS-CoV-2`

If you reorder `DISEASE_MAP` or care about a different precedence for these 5 rows, branch on the full `disease_categories_for` list instead of relying on the `cats[-1]` tie-break.

In [ ]:
TARGET_LABELS = [
    "IPF / Pulmonary Fibrosis",
    "COVID-19 / SARS-CoV-2",
    "COPD",
    "Interstitial Lung Disease",
    "Cystic Fibrosis",
]
QC_THRESHOLDS = QcThresholds(
    minMedianGenesPerCell=1000,
    # minMedianUmisPerCell=2000,  # optional, uncomment to enable
)

lung = result.lungIntersection.copy()
lung["diseaseLabel"] = lung["disease"].apply(most_specific_disease_label)
selected = lung[lung["diseaseLabel"].isin(TARGET_LABELS)]

qc_df = compute_obs_qc(selected["srx_accession"].tolist(), cfg)
qc_passed = apply_qc(selected, qc_df, QC_THRESHOLDS)

print("per-label counts (selected -> after QC):")
selected_counts = selected.groupby("diseaseLabel").size().reindex(TARGET_LABELS, fill_value=0)
qc_counts = qc_passed.groupby("diseaseLabel").size().reindex(TARGET_LABELS, fill_value=0)
for label in TARGET_LABELS:
    print(f"  {label:32s}  selected={selected_counts[label]:4d}  passed QC={qc_counts[label]:4d}")

In [ ]:
SAMPLE_SIZE = 25
RANDOM_STATE = 42

sampled = pd.concat(
    [
        qc_passed[qc_passed["diseaseLabel"] == label].sample(
            n=min(SAMPLE_SIZE, int((qc_passed["diseaseLabel"] == label).sum())),
            random_state=RANDOM_STATE,
        )
        for label in TARGET_LABELS
    ],
    ignore_index=True,
)

OUTPUT_COLUMNS = [
    "srx_accession",
    "file_path",
    "obs_count",
    "disease",
    "diseaseLabel",
    "medianGenesPerCell",
    "medianUmisPerCell",
    "nCellsForQc",
]
cfg.outputDir.mkdir(parents=True, exist_ok=True)
csv_path = cfg.outputDir / "datasets_subset_qc.csv"
sampled[OUTPUT_COLUMNS].to_csv(csv_path, index=False)

final_counts = sampled.groupby("diseaseLabel").size().reindex(TARGET_LABELS, fill_value=0)
for label in TARGET_LABELS:
    print(f"  {label:32s}  passed QC={qc_counts[label]:4d}  kept={final_counts[label]:4d}")
print(f"\nwrote {csv_path} ({len(sampled):,} accessions)")

## Spot Check

In [ ]:
biggest_srx = result.lungIntersection.loc[result.lungIntersection["obs_count"].idxmax(), "srx_accession"]
print(f"largest SRX: {biggest_srx} ({result.lungIntersection['obs_count'].max():,} cells)")
obs_rows_for_srx(biggest_srx, cfg)["cell_type"].value_counts()